In [ ]:
# 03_metrics_reliability_dei.ipynb
# From the cached recourse, compute the three-constraint reliability (immutability,
# action horizon, non-controllability), the recourse-rate table, the failure
# decomposition, the full-population reliability with exact bounds, and the Double
# Exclusion Index as an exact joint probability (chain rule, hard-decision rejection
# rate). All result tables are written to results/tables. No identifying paths printed.

from pathlib import Path
import numpy as np, pandas as pd, pickle
from math import sqrt
from scipy.stats import beta

ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
CACHE = ROOT / "results" / "cache"; TAB = ROOT / "results" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
TAU = 0.20

A = pickle.load(open(CACHE / "nb1_artifacts.pkl", "rb"))
rec = pickle.load(open(CACHE / "recourse_main.pkl", "rb"))
disfull = pickle.load(open(CACHE / "recourse_dis_full.pkl", "rb"))
frames = pickle.load(open(CACHE / "nb2_frames.pkl", "rb")); dis = frames["dis_full"]
rf = A["rf"]; FEAT = A["FEATURES"]; IMMUT = A["IMMUTABLE"]; ACT = A["ACTIONABLE"]; NON = A["NON_CONTROLLABLE"]
Xte = A["X_test"]; group_test = A["group_test"]

def flags(o, c):
    imm = any(not np.isclose(float(o[f]), float(c[f])) for f in IMMUT)
    act = False
    for f in ACT:
        a = float(o[f]); b = float(c[f]); act |= (b != 0) if a == 0 else abs(b - a) / abs(a) > TAU + 1e-9
    non = any(not np.isclose(float(o[f]), float(c[f])) for f in NON)
    return imm, act, non

def score(sample, cfl):
    n = len(sample); rr = rel = pt = pr = 0
    for (_, o), cfs in zip(sample.iterrows(), cfl):
        if cfs is None or len(cfs) == 0: continue
        rr += 1; ok = False
        for _, c in cfs.iterrows():
            i, a, nn = flags(o, c); pt += 1
            if not i and not a and not nn: pr += 1; ok = True
        if ok: rel += 1
    return dict(RR=100*rr/n, PR=100*pr/pt if pt else 0.0, BR=100*rel/n, n=n, nrel=rel)

def cp_upper(k, n, a=0.05): return 1.0 if k == n else beta.ppf(1 - a/2, k + 1, n - k)
def wilson(k, n, z=1.96):
    p = k/n; den = 1 + z*z/n; cen = (p + z*z/(2*n))/den; half = z*sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return 100*(cen-half), 100*(cen+half)

# Table 1: recourse rates (three-constraint reliability).
rows = []
for g in ["advantaged", "disadvantaged"]:
    s = rec[g]["_sample"]
    for scn in ["A", "B", "C"]:
        r = score(s, rec[g][scn]); rows.append({"group": g, "scenario": scn,
            "RR_pct": round(r["RR"],1), "PR_RR_pct": round(r["PR"],2), "BR_RR_pct": round(r["BR"],2)})
pd.DataFrame(rows).to_csv(TAB / "table1_recourse_rates.csv", index=False)

# Table 2: failure decomposition by constraint (Scenarios A, B).
rows = []
for g in ["advantaged", "disadvantaged"]:
    s = rec[g]["_sample"]
    for scn in ["A", "B"]:
        pt = im = ac = no = 0
        for (_, o), cfs in zip(s.iterrows(), rec[g][scn]):
            if cfs is None or len(cfs) == 0: continue
            for _, c in cfs.iterrows():
                i, a, nn = flags(o, c); pt += 1; im += i; ac += a; no += nn
        pt = pt or 1
        rows.append({"group": g, "scenario": scn, "P_imm_pct": round(100*im/pt,1),
            "P_act_pct": round(100*ac/pt,1), "P_non_pct": round(100*no/pt,1)})
pd.DataFrame(rows).to_csv(TAB / "table2_failure_breakdown.csv", index=False)

# Table 3: full disadvantaged pool with exact bounds.
rows = []
for scn in ["A", "B", "C"]:
    r = score(dis, disfull[scn]); rows.append({"scenario": scn, "RR_pct": round(r["RR"],2),
        "n_reliable": r["nrel"], "BR_RR_pct": round(r["BR"],3), "upper95_pct": round(100*cp_upper(r["nrel"], r["n"]),3)})
pd.DataFrame(rows).to_csv(TAB / "table3_fullpool_reliability.csv", index=False)

# Advantaged Scenario-C interval (bootstrap + Wilson) for reference.
s = rec["advantaged"]["_sample"]; flagsC = []
for (_, o), cfs in zip(s.iterrows(), rec["advantaged"]["C"]):
    ok = 0
    if cfs is not None and len(cfs):
        for _, c in cfs.iterrows():
            i, a, nn = flags(o, c)
            if not i and not a and not nn: ok = 1; break
    flagsC.append(ok)
flagsC = np.array(flagsC); rng = np.random.default_rng(42)
boot = np.array([flagsC[rng.integers(0, len(flagsC), len(flagsC))].mean() for _ in range(2000)])
wl, wh = wilson(int(flagsC.sum()), len(flagsC))
pd.DataFrame([{"estimate_pct": round(100*flagsC.mean(),2),
    "bootstrap_lo_pct": round(100*np.percentile(boot,2.5),2), "bootstrap_hi_pct": round(100*np.percentile(boot,97.5),2),
    "wilson_lo_pct": round(wl,2), "wilson_hi_pct": round(wh,2)}]).to_csv(TAB / "table3b_advantaged_C_interval.csv", index=False)

# Table 5: Double Exclusion Index (exact joint; empirical hard-rejection first factor).
pred = rf.predict(Xte[FEAT]); dfp = pd.DataFrame({"group": group_test.values, "pred": pred})
brrc = {r["group"]: r["BR_RR_pct"] for r in pd.read_csv(TAB / "table1_recourse_rates.csv").query("scenario=='C'").to_dict("records")}
brrc["disadvantaged"] = pd.read_csv(TAB / "table3_fullpool_reliability.csv").query("scenario=='C'")["BR_RR_pct"].iloc[0]
rows = []
for g in ["advantaged", "disadvantaged"]:
    prej = (dfp[dfp.group == g]["pred"] == 0).mean(); norec = 1 - brrc[g]/100
    rows.append({"group": g, "P_reject": round(prej,4), "P_no_recourse": round(norec,4), "DEI": round(prej*norec,4)})
pd.DataFrame(rows).to_csv(TAB / "table5_dei.csv", index=False)

print("Notebook 03 complete. Recourse-rate, failure-decomposition, full-pool, interval, and DEI tables written.")
